<a href="https://colab.research.google.com/github/kevin305-dev/hello-world/blob/main/Prompt_Templates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain
!pip install openai

In [2]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [ ]:
#import os
#import openai
#openai.api_key = os.environ["OPENAI_API_KEY"]

In [3]:
from langchain_openai.llms import OpenAI
from langchain_openai.chat_models import ChatOpenAI

Colab Secrets에서 OPENAI_API_KEY 가져오기

In [4]:
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

In [26]:
davinci3_params = {
    "model_name" : "gpt-3.5-turbo",
    "max_tokens" : 1000
}

아래 코드로 OPENAI_API_KEY와


*   1) 일반 프롬프트 템플릿을 위한 모델 "text-davinci-003"
*   2) 챗 전용 프롬프트 템플릿을 위한 모델 "gpt-3.5-turbo"를 선언한다





In [ ]:
davinci3 = OpenAI(
    model_name = "text-davinci-003",
    openai_api-key = openai_api_key,
    max_tokens = 1000
)

In [ ]:
davinci3 = ChatOpenAI(
    openai_api_key=openai_api_key,
    model = "gpt-3.5-turbo",
    max_tokens = 1000
)

PromtTemplate 맛 보기
2가지 종류
1. PromptTemplates, 일반적인 LLM
2. ChatPromptTemplates, 챗팅 특화 LLM

1.PromptTemplates는 일반적인 프롬프트 템플릿 생성시 활용

2.ChatpromptTemplates는 채팅특화 LLM에 프롬프트를 전달하는 데 활용하는 특화 프롬프트 템플릿

In [6]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
string_prompt = PromptTemplate.from_template("Tell me a joke about {topic}")
string_prompt_value = string_prompt.format_prompt(topic="soccer")
string_prompt_value

StringPromptValue(text='Tell me a joke about soccer')

In [7]:
print(string_prompt_value.to_string())

Tell me a joke about soccer


In [8]:
chat_prompt = ChatPromptTemplate.from_template("Tell me a joke about {subject}")
chat_prompt_value = chat_prompt.format_prompt(subject="soccer")
chat_prompt_value

ChatPromptValue(messages=[HumanMessage(content='Tell me a joke about soccer', additional_kwargs={}, response_metadata={})])

In [9]:
chat_prompt_value.to_string()

'Human: Tell me a joke about soccer'

# **(2) 프롬프트 템플릿 활용해 보기**

반복적인 프롬프트를 삽입해야하는 경우, Prompt Templates를 통해 간편하게 LLM을 활용할 수 있습니다.


*   davinci3 모델의 일반 프롬프트 템플릿과
*   GPT-3 모델의 챗팅 특화 프롬프트 템플릿
을 활용하여 대화해 보기   

### (2-1) 일반 프롬프트 템플릿
[2-1-1] 프롬프트 템플릿 만들기
   
*   템플릿 = template
*   input_variables에 List형태로 "재료"를 담아준다
*   프롬프트 템플릿 = 만든 template을 넣어준다



In [11]:
from langchain_core.prompts.prompt import PromptTemplate
template = """
너는 요리사야. 내가 가진 재료들을 갖고 만들 수 있는 요리를 추천하고, 그 요리의 레시피를 제시해줘.
내가 가진 재료는 아래와 같아.

<재료>
{재료}

"""
prompt_template = PromptTemplate(
    input_variables=["재료"],
    template=template
)


*   prompt_template에 '재료'를 입력변수로 제공한다
*   사용자는 가변적인 '재료'만 넣어서 원하는 결과를 얻을 수 있다.
*   프롬프트를 간편화 시킴.
*   프롬프트 템플릿 = 서비스를 만들때 사용자가 입력해야하는 부분의 부담을 덜어주는 역할을 한다

*   입력 값을 넣어, 만들어진 '일반 프롬프트 템플릿'이 잘되었는지 출력하여 점검

In [13]:
print(prompt_template.format(재료='양파,계란,사과,빵'))


너는 요리사야. 내가 가진 재료들을 갖고 만들 수 있는 요리를 추천하고, 그 요리의 레시피를 제시해줘.
내가 가진 재료는 아래와 같아.

<재료>
양파,계란,사과,빵




[2-1-2] 일반 프롬프트 템플릿을 모델에 넣어 결과 확인
*   만들어진 '일반 프롬프트 템플릿'과
*   입력 값인 '재료'를
*   모델 "text-davinci-003"를 이용하는 'davinci3'에 넣어 결과확인



In [30]:
print(davinci3.invoke(
    prompt_template.format(재료='양파,계란,사과,빵')
    )
)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

### (2-2) 챗팅 특화 프롬프트 템플릿
[2-2-1] ChatGPT와 프롬프트 템플릿을 활용하여 대화해보기

In [32]:
!pip install langchain_core

*   SystemMessage : 제공되는 템플릿
*   HumanMessage : 입력되는 '재료'
*   AIMessage : 출력 생성되는 답변

In [36]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

In [37]:
# ChatGPT 모델을 로드
chatgpt = ChatOpenAI(temperature=0)

# ChatGPT에 역할 부여
# 위에서 정의한 템플릿(template) 사용
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

# 사용자가 정의한 매개변수 정의
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template="{재료}")
# human_template = "{재료}"
# human_message_prompt = HumanMessagePromptTemplate.from_template(human_template) 와 동일 표기

# 챗팅 특화 프롬프트 템플릿(ChatPromptTemplate)에
# system message와 human message를 삽입한다
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# ChatGPT API에 만들어진 챗팅 특화 프롬프트 템플릿(ChatPromptTemplate)을 입력할때,
# human message의 매개변수 '재료'를 할당하여 전달한다
# ==> 이 방식을 통해 ChatGPT는 챗팅 특화 프롬프트 템플릿(ChatPromptTemplate)의 구성요소인 system message, human message를 전달받아, 대답 생성에 활용한다
answer = chatgpt(chat_prompt.format_prompt(재료='양파,계란,사과,빵').to_messages())

print(answer.content)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

### [Key] 챗팅 모델의 경우

*   프롬프트 템플릿도
*   챗팅 모델의 형식(SystemMessage,AIMessage,HumanMessage)에 맞춤



# **(3) Few-shot 예제를 통한 프롬프트 템플릿**

### Few-shot

*   딥러닝 모델이 결과물을 출력할때, 예시 결과물을 제시함으로써 원하는 결과물을 유도하는 방식
*   LLM 역시, Few-shot 예제를 제공하면 예제와 유사한 형태의 결과물을 출력한다
* 내가 원하는 결과물의 형태가 특수하거나, 구조화된 답변을 원할 경우, 예시를 여러개 제시하면 결과물의 품질을 향상시킬 수 있다



In [ ]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

examples = [
    {
        "question" : "아이유로 삼행시 만들어줘",
        "answer" :
        """
        아 : 아이유는
        이 : 이런 강의를 들을 이
        유 : 유가 없다.
        """
    },
    {
        "question" : "김민수로 삼행시 만들어줘",
        "answer" :
        """
        김 : 김치는 맛있다
        민 : 민달팽이도 좋아하는 김치
        수 : 수억을 줘도 김치는 내꺼!!!
        """
    }
]

-  example prompt에 만든 예시를 넣어줌
-  첫번째 예제인 '아이유' 삼행시 출력

In [ ]:
example_prompt = PromptTemplate(input_variables=["question","answer"],template="Question: {question}\n{answer}")

print(example_prompt.format(**examples[0])

- Few-shot 프롬프트 템플릿을 선언

In [ ]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"]
)

print(prompt.fromat(input="호날두로 삼행시 만들어줘"))

In [ ]:
print(davinci3.predict("호날두로 삼행시 만들어줘"))